# Perbandingan Model Klasifikasi Rekomendasi Jurusan (EduPath AI / JurusanKu ID)

Notebook ini dibuat untuk melatih beberapa model Machine Learning (Decision Tree, Random Forest, dan Gradient Boosting), membandingkan performanya (akurasi), serta mengekspor model terbaik ke dalam format `model.pkl` yang kompatibel dengan aplikasi Django JurusanKu ID.

## 1. Import Library & Upload Dataset

Silakan jalankan cell di bawah ini. Anda dapat mengunggah file `dataset.csv` dari folder `ml_model/` project Anda ke panel Google Colab (klik ikon folder di sebelah kiri, lalu drag-and-drop file `dataset.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

print("Library berhasil di-import!")

## 2. Membaca dan Mengeksplorasi Dataset

In [ ]:
# Membaca file dataset.csv (menggunakan separator semicolon ';')
# Pastikan file 'dataset.csv' sudah di-upload ke Colab
if os.path.exists('dataset.csv'):
    df = pd.read_csv('dataset.csv', sep=';')
    # Membersihkan kolom kosong jika ada
    df = df.dropna(axis=1, how='all')
    df.columns = df.columns.str.strip()
    print("Dataset berhasil dimuat!")
    print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom\n")
    print("5 Baris Pertama:")
    display(df.head())
else:
    print("ERROR: File 'dataset.csv' tidak ditemukan. Silakan upload file dataset.csv ke sidebar Colab terlebih dahulu!")

## 3. Preprocessing & Split Data (70:30)

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df.drop('jurusan', axis=1)
y = df['jurusan']

# Split dataset menjadi 70% Train dan 30% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Jumlah data training: {X_train.shape[0]} sampel")
print(f"Jumlah data testing: {X_test.shape[0]} sampel")

## 4. Melatih Beberapa Model Klasifikasi

In [ ]:
# 1. Decision Tree
dt_model = DecisionTreeClassifier(max_depth=7, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

# 3. Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_acc = accuracy_score(y_test, gb_preds)

print("Training selesai!")

## 5. Perbandingan Performa Model

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Gradient Boosting'],
    'Akurasi (%)': [dt_acc * 100, rf_acc * 100, gb_acc * 100]
})

print("=== TABEL PERBANDINGAN AKURASI ===")
display(comparison_df.sort_values(by='Akurasi (%)', ascending=False).reset_index(drop=True))

# Visualisasi Perbandingan
plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Akurasi (%)', data=comparison_df, palette='viridis')
plt.title('Perbandingan Akurasi Model Klasifikasi')
plt.ylim(0, 100)
for index, row in comparison_df.iterrows():
    plt.text(index, row['Akurasi (%)'] + 1, f"{row['Akurasi (%)']:.2f}%", color='black', ha="center")
plt.show()

## 6. Menyimpan Model Terbaik ke model.pkl

In [ ]:
# Menentukan model dengan akurasi tertinggi
models = {
    'Decision Tree': (dt_model, dt_acc),
    'Random Forest': (rf_model, rf_acc),
    'Gradient Boosting': (gb_model, gb_acc)
}

best_model_name = max(models, key=lambda k: models[k][1])
best_model, best_acc = models[best_model_name]

print(f"Model terbaik adalah: {best_model_name} dengan Akurasi: {best_acc:.2%}\n")

# Menyusun data model dengan format dictionary agar kompatibel dengan views.py Django
model_data = {
    'model': best_model,
    'decision_tree': best_model, # Fallback agar views.py tidak error jika mencari key ini
    'accuracies': {
        'decision_tree': float(dt_acc),
        'random_forest': float(rf_acc),
        'gradient_boosting': float(gb_acc)
    },
    'dataset_size': len(df)
}

# Simpan ke model.pkl
joblib.dump(model_data, 'model.pkl')
print("Model berhasil diekspor ke file 'model.pkl'!")
print("Silakan download file 'model.pkl' dari panel sebelah kiri Google Colab dan letakkan ke folder 'ml_model/' di project Django Anda.")